# TG 7-to-1 |Y⟩ State Distillation

Transversal-Gate 7-to-1 magic state distillation using the PQRM [[7,1,3]] (hypercube) encoding.

## Protocol overview (Zhou et al. Fig 4c)

16 patches total — 8 "magic" patches (M1–M7 + W0) + 8 "working" patches:

```
Encoding layer: 8 working patches W0, W1, ..., W7 form the [[7,1,3]] PQRM codeblock
Magic inputs:   M1, M2, ..., M7 supply noisy |Y⟩ states
```

Steps:
1. **Init**: prepare M1–M7 with noisy |Y⟩ = fold-transversal S; W0–W7 |0⟩_L
2. **SE(d)** stabiliser extraction on all patches
3. **Hypercube encoding**: 3 ticks of transversal CNOTs (encode W0..W7 into PQRM codeblock), SE between ticks
4. **Teleportation**: CNOT(W_i, M_i) for i=1..7 → SE → H on M1..M7 → SE → measure M1..M7 in Z
5. **Readout**: noiseless S†_L + H on W0 → MZ

## |Y⟩ state preparation method

Same as LS distillation: **`fold_transversal_s`** on magic patches M1–M7.  
NOT corner/middle injection — the code distance protects against errors in the S gate.

## p_in / p_out

- **p_in**: logical error rate of one noisy |Y⟩ from `estimate_p_in()` (same calibration as LS)
- **p_out**: post-selected LER of W0 output (post-select on M1–M7 syndromes)
- **Theory**: p_out ≈ 7 · p_in³ (same as Steane, same order)

Code: Unrotated Surface Code, d=3, rounds=1 (demo).

In [ ]:
import sys
from pathlib import Path
import io, contextlib

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from benchmarks.logical_circuits.distillation.tg_7to1.TG_distillation_7_to_1 import (
    build_distillation_circuit,
    inject_noise,
    estimate_p_in,
    run_simulation,
    _TG_MAGIC_NAMES,
)
from lightstim.simulation.observable_analysis import (
    build_obs_patch_matrix,
    identify_distillation_observables,
)

In [ ]:
d = 3
rounds = 1   # use rounds=d for paper benchmark

## 1. Build Circuit

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    circuit, circuit_info, system = build_distillation_circuit(d, rounds, r=1)
print(f"d={d}: {circuit_info['num_qubits']} qubits, {circuit_info['num_detectors']} detectors, {circuit_info['num_observables']} observables")

## 2. Noiseless Sanity Check

In [ ]:
import numpy as np
dets, obs = circuit.compile_detector_sampler().sample(shots=500, separate_observables=True)
print(f"Noiseless: det errors={np.any(dets)}, obs errors={np.any(obs)}")

## 3. Observable Analysis

In [ ]:
matrix, patch_names = build_obs_patch_matrix(circuit, system)
print(f"Patches: {patch_names}")
_, target_obs, ps_obs = identify_distillation_observables(matrix, patch_names, ["W0"])
print(f"Target obs (W0 output): {target_obs}")
print(f"Post-select obs (M1..M7): {ps_obs}")

## 4. p_in Calibration

Same calibration method as LS: RX → noisy S → SE(rounds) → noiseless S† → MX.

In [ ]:
p_injected_values = [1e-3, 5e-3, 2e-2]
p_in_values = {}

for p_inj in p_injected_values:
    p_in = estimate_p_in(d, rounds, r=1, p_injected=p_inj,
                         max_shots=500_000, max_errors=50, batch_size=5_000)
    p_in_values[p_inj] = p_in
    print(f"p_injected={p_inj:.0e}  →  p_in={p_in:.3e}")

## 5. Distillation Simulation (injection-only noise)

In [ ]:
magic_qubits = {q for q, owner in system.index_to_owner_map.items()
                if owner in _TG_MAGIC_NAMES}
magic_data_qubits = magic_qubits & system.data_indices

results = []
for p_inj in p_injected_values:
    print(f"\np_injected={p_inj:.0e}  (p_in≈{p_in_values[p_inj]:.2e})")
    stats = run_simulation(
        circuit, magic_qubits, p=0.0, p_injected=p_inj, mode="injection",
        ps_obs=ps_obs, target_obs=target_obs,
        decoder_name="pymatching",
        max_shots=500_000, max_errors=30, batch_size=5_000,
        data_indices=magic_data_qubits,
    )
    results.append({
        'p_inj': p_inj,
        'p_in': p_in_values[p_inj],
        'p_out': stats.logical_error_rate,
        'ps_rate': stats.post_selection_rate,
        'shots': stats.shots,
        'errors': stats.errors,
    })
    print(f"  p_out={stats.logical_error_rate:.2e}  PS_rate={stats.post_selection_rate:.2f}  ({stats.errors}/{stats.shots:,})")

## 6. P_out vs P_in Plot

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))

p_in_arr = [r['p_in'] for r in results if r['errors'] >= 5]
p_out_arr = [r['p_out'] for r in results if r['errors'] >= 5]

if p_in_arr:
    ax.loglog(p_in_arr, p_out_arr, 's-', color='darkorange', lw=2, ms=8,
              markeredgecolor='k', markeredgewidth=0.4, label=f'd={d}')
    pin_th = np.logspace(np.log10(min(p_in_arr)) - 0.5, np.log10(max(p_in_arr)) + 0.3, 200)
    ax.loglog(pin_th, 7 * pin_th**3, 'k--', lw=1.8, label=r'$7\,P_{\rm in}^3$')

ax.set_xlabel(r'$P_{\rm in}$', fontsize=13)
ax.set_ylabel(r'$P_{\rm out}$', fontsize=13)
ax.set_title(f'TG 7-to-1 Distillation — d={d}', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, which='both', ls='--', alpha=0.4)
plt.tight_layout()
plt.show()